In [1]:
import numpy as np 
import pandas as pd

gsv = pd.read_csv("gsv_factors.csv", index_col = "policy_year")
gsv.head()

,pt_10,pt_12,pt_15,pt_20
policy_year,,,,
1,NaN,NaN,NaN,NaN
2,30.0,30.0,30.0,30.0
3,35.0,35.0,35.0,35.0
4,50.0,50.0,50.0,50.0
5,50.0,50.0,50.0,50.0


In [7]:
#Inputs and Validation

ap = 50000
ppt = 10
pt = 15
sam = 750000

#Valid PPT/PT combinations
allowed = {
    5: [10, 12, 15, 20], 6: [10, 12, 15, 20], 7: [10, 12, 15, 20],
    8: [12, 15, 20], 9: [12, 15, 20], 10: [12, 15, 20],
    12: [15, 20],
}

assert 5000 <= ap <= 250000, "Premium outside allowed limits"
assert ppt in allowed and pt in allowed[ppt], "Invalid PPT / PT combination"

In [15]:
t = pd.DataFrame({"policy_year": range(1, pt + 1)})

#To stop premiums after ppt
t["premiums_paid_count"] = t["policy_year"].clip(upper = ppt)
t["total_premiums_paid"] = ap * t["premiums_paid_count"]

#Computing death benefit (Higher of 3)
t["db_10x_ap"] = 10 * ap
t["db_105pct_tpp"] = 1.05 * t["total_premiums_paid"]
t["db_sam"] = sam
t["death_benefit"] = t[["db_10x_ap", "db_105pct_tpp", "db_sam"]].max(axis = 1)

#Computing GSV = factor * total premiums paid
t["gsv_factor_pct"] = t["policy_year"].map(gsv[f"pt_{pt}"])
t["gsv"] = (t["gsv_factor_pct"].fillna(0) / 100 * t["total_premiums_paid"]).round(0)

#Computing Maturity benefit (payable only in the last year)
t["maturity_benefit"] = np.where(t["policy_year"] == pt, sam, 0)

summary = t[["policy_year", "total_premiums_paid", "death_benefit", "gsv", "maturity_benefit"]]
summary

,policy_year,total_premiums_paid,death_benefit,gsv,maturity_benefit
0,1,50000,750000.0,0.0,0
1,2,100000,750000.0,30000.0,0
2,3,150000,750000.0,52500.0,0
3,4,200000,750000.0,100000.0,0
4,5,250000,750000.0,125000.0,0
5,6,300000,750000.0,150000.0,0
6,7,350000,750000.0,175000.0,0
7,8,400000,750000.0,224000.0,0
8,9,450000,750000.0,274500.0,0
9,10,500000,750000.0,335000.0,0
